# PointsX — YOLO-seg Training (Synthetic Silhouettes)

Uses the **same** SMPL-X bodies + camera positions as the pose dataset, but renders white-on-black silhouettes via an emission shader. Converts masks to YOLO polygon labels, then finetunes `yolo11n-seg`.

**Input:** Same manifest as pose pipeline (`manifest.json` from the SMPL-X phase).
**Output:** `yolo11n-seg-pointsx.pt` — trained for 1-class body segmentation.

> You can run this **after** generating `manifest.json` in `03_generate_synthetic_colab.ipynb` (SMPL-X phase only; no need to run the photo renders first).

Expected runtime on T4 GPU: **~1 h** for 1500 masks + training.

## 0. Configuration

In [ ]:
DRIVE_ROOT   = "/content/drive/MyDrive/PointsX"
BLENDER_JOBS = 2
USE_GPU      = True
USE_CYCLES   = True       # Cycles w/ 1 sample is plenty for solid emission renders

# Chunk range for multi-account splitting (None = all)
CHUNK_START  = None
CHUNK_END    = None

WORK_DIR     = "/content/pointsx_work"
ASSETS_DIR   = f"{WORK_DIR}/assets"              # not really needed for masks, kept for CLI compat
SEG_OUT_DIR  = f"{WORK_DIR}/data/synthetic-seg"  # local render + label output
DRIVE_SEG    = f"{DRIVE_ROOT}/data/synthetic-seg"

# Re-use the manifest from the pose pipeline
POSE_OUT_DIR = f"{WORK_DIR}/data/synthetic-pose"
DRIVE_POSE   = f"{DRIVE_ROOT}/data/synthetic-pose"

print("Config OK")

## 1. Mount Drive + install Blender + PointsX

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for d in (WORK_DIR, ASSETS_DIR, SEG_OUT_DIR, POSE_OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Directories created.")

In [ ]:
%%bash
# Blender
if [ ! -f /content/blender/blender ]; then
    wget -q https://download.blender.org/release/Blender3.6/blender-3.6.9-linux-x64.tar.xz -O /tmp/blender.tar.xz
    tar -xf /tmp/blender.tar.xz -C /content/
    mv /content/blender-3.6.9-linux-x64 /content/blender
fi
/content/blender/blender --version | head -1

In [ ]:
BLENDER_DIR = "/content/blender"
BLENDER_EXE = f"{BLENDER_DIR}/blender"

import subprocess
blender_pip = f"{BLENDER_DIR}/3.6/python/bin/python3.10"
subprocess.run([blender_pip, "-m", "pip", "install", "-q", "numpy", "rich"], check=False)
print("Blender Python packages ready.")

In [ ]:
%%bash
set -e
REPO_GIT="https://github.com/feed7362/PointsX.git"
rm -rf /content/pointsx_repo
git clone -q "$REPO_GIT" /content/pointsx_repo
cd /content/pointsx_repo && pip install -q -e .
pip install -q opencv-python rich tqdm ultralytics
echo "PointsX installed."

## 2. Get the manifest

If you already ran the SMPL-X phase on this Drive, the manifest + meshes are there. Otherwise the pose training Drive folder may not exist and you need to run notebook 03 first.

In [ ]:
import subprocess, os, json
from pathlib import Path

# Restore the pose working dir (contains manifest.json + meshes + landmarks)
if Path(DRIVE_POSE).exists():
    print("Restoring pose working dir from Drive (this may take a few minutes)...")
    subprocess.run(["rsync", "-a", f"{DRIVE_POSE}/meshes/",    f"{POSE_OUT_DIR}/meshes/"], check=True)
    subprocess.run(["rsync", "-a", f"{DRIVE_POSE}/landmarks/", f"{POSE_OUT_DIR}/landmarks/"], check=True)
    subprocess.run(["rsync", "-a", f"{DRIVE_POSE}/manifest.json", f"{POSE_OUT_DIR}/"], check=True)
    print("Done.")
else:
    raise RuntimeError(
        f"{DRIVE_POSE} not found on Drive. Run notebook 03 first to generate the SMPL-X phase."
    )

manifest_path = Path(POSE_OUT_DIR) / "manifest.json"
manifest = json.loads(manifest_path.read_text())
print(f"Manifest: {len(manifest)} entries")

## 3. Test mask render (1 body, verify scene)

In [ ]:
import subprocess, json, os
from pathlib import Path

test_manifest = [manifest[0]] if manifest else []
test_manifest_path = Path(SEG_OUT_DIR) / "test_manifest.json"
test_manifest_path.parent.mkdir(parents=True, exist_ok=True)
test_manifest_path.write_text(json.dumps(test_manifest, indent=2))

env = {**os.environ, "PYTHONPATH": "/content/pointsx_repo/src"}
engine = "CYCLES" if USE_CYCLES else "BLENDER_EEVEE"
blender_script = "/content/pointsx_repo/src/pointsx/synthetic/blender_render.py"

cmd = [
    BLENDER_EXE, "--background", "--python", blender_script, "--",
    "--manifest", str(test_manifest_path),
    "--out-dir",  SEG_OUT_DIR,
    "--assets",   ASSETS_DIR,
    "--engine",   engine,
    "--mode",     "mask",
    "--start-idx", "0", "--end-idx", "1",
]
if USE_GPU:
    cmd.append("--gpu")

print("Running test mask render...")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300, env=env)
print("STDOUT:", result.stdout[-2000:])
if result.stderr:
    print("STDERR:", result.stderr[-1500:])

for split in ("train", "val"):
    imgs = list(Path(SEG_OUT_DIR, split, "images").glob("s00001_*.jpg"))
    if imgs:
        print(f"  {split}: {[p.name for p in imgs]}")

In [ ]:
# Preview the test mask
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

for split in ("train", "val"):
    for img_path in sorted(Path(SEG_OUT_DIR, split, "images").glob("s00001_*.jpg")):
        plt.figure(figsize=(4, 4))
        plt.imshow(Image.open(img_path), cmap="gray")
        plt.title(img_path.name)
        plt.axis("off")
        plt.show()

## 4. Full mask render (parallel + rsync + resume)

In [ ]:
import json, subprocess, os, math, time
from pathlib import Path

# Resume: pull any previously rendered masks from Drive
if Path(DRIVE_SEG).exists():
    print("Restoring previous mask renders from Drive...")
    subprocess.run(["rsync", "-a", f"{DRIVE_SEG}/", f"{SEG_OUT_DIR}/"], capture_output=True)
    n = len(list(Path(SEG_OUT_DIR, "train", "images").glob("*.jpg"))) + \
        len(list(Path(SEG_OUT_DIR, "val", "images").glob("*.jpg")))
    print(f"  Restored {n} existing images.")
os.makedirs(DRIVE_SEG, exist_ok=True)

start_idx = CHUNK_START if CHUNK_START is not None else 0
end_idx   = CHUNK_END   if CHUNK_END   is not None else len(manifest)
chunk_manifest_path = Path(SEG_OUT_DIR) / "manifest.json"
chunk_manifest_path.write_text(json.dumps(manifest, indent=2))

engine = "CYCLES" if USE_CYCLES else "BLENDER_EEVEE"
blender_script = "/content/pointsx_repo/src/pointsx/synthetic/blender_render.py"
env = {**os.environ, "PYTHONPATH": "/content/pointsx_repo/src"}

total = end_idx - start_idx
jobs = min(BLENDER_JOBS, max(total, 1))
chunk = math.ceil(total / jobs)
print(f"Rendering masks: entries {start_idx}–{end_idx} | {jobs} parallel job(s)")

procs = []
for j in range(jobs):
    s = start_idx + j * chunk
    e = min(s + chunk, end_idx)
    if s >= end_idx:
        break
    cmd = [
        BLENDER_EXE, "--background", "--python", blender_script, "--",
        "--manifest", str(chunk_manifest_path),
        "--out-dir",  SEG_OUT_DIR,
        "--assets",   ASSETS_DIR,
        "--engine",   engine,
        "--mode",     "mask",
        "--start-idx", str(s), "--end-idx", str(e),
    ]
    if USE_GPU:
        cmd.append("--gpu")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    procs.append((j, proc))
    print(f"  Job {j}: entries {s}–{e}")

last_sync = time.time()
SYNC_INTERVAL = 180  # rsync every 3 min
for j, proc in procs:
    for line in proc.stdout:
        if "[blender_render]" in line or "ERROR" in line or "Done" in line:
            print(f"[job-{j}] {line}", end="")
        if time.time() - last_sync > SYNC_INTERVAL:
            subprocess.Popen(["rsync", "-a", "--update", f"{SEG_OUT_DIR}/train/", f"{DRIVE_SEG}/train/"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.Popen(["rsync", "-a", "--update", f"{SEG_OUT_DIR}/val/",   f"{DRIVE_SEG}/val/"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            last_sync = time.time()
    proc.wait()

print("\nFinal sync to Drive...")
subprocess.run(["rsync", "-a", "--update", f"{SEG_OUT_DIR}/train/", f"{DRIVE_SEG}/train/"], capture_output=True)
subprocess.run(["rsync", "-a", "--update", f"{SEG_OUT_DIR}/val/",   f"{DRIVE_SEG}/val/"],   capture_output=True)

n_local = len(list(Path(SEG_OUT_DIR, "train", "images").glob("*.jpg"))) + \
          len(list(Path(SEG_OUT_DIR, "val",   "images").glob("*.jpg")))
print(f"Total mask images: {n_local}")

## 5. Convert silhouette PNGs → YOLO polygon labels

In [ ]:
!python -m pointsx.synthetic.masks_to_polygons --data {SEG_OUT_DIR}

In [ ]:
# Sanity-check: overlay a polygon on the silhouette
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img_dir   = Path(SEG_OUT_DIR) / "train" / "images"
label_dir = Path(SEG_OUT_DIR) / "train" / "labels"

samples = random.sample(sorted(img_dir.glob("*.jpg")), min(4, len(list(img_dir.glob("*.jpg")))))
fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
if len(samples) == 1:
    axes = [axes]
for ax, img_path in zip(axes, samples):
    img = np.array(Image.open(img_path))
    ax.imshow(img, cmap="gray")
    lbl = label_dir / img_path.with_suffix(".txt").name
    if lbl.exists():
        parts = lbl.read_text().split()
        coords = np.array(parts[1:], dtype=float).reshape(-1, 2)
        H, W = img.shape[:2]
        xs = coords[:, 0] * W
        ys = coords[:, 1] * H
        ax.plot(np.append(xs, xs[0]), np.append(ys, ys[0]), color="lime", linewidth=1.5)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Train YOLO11n-seg

In [ ]:
# Sync labels + dataset.yaml to Drive so we don't lose them if Colab disconnects
!rsync -a --update {SEG_OUT_DIR}/train/labels/ {DRIVE_SEG}/train/labels/
!rsync -a --update {SEG_OUT_DIR}/val/labels/   {DRIVE_SEG}/val/labels/
!cp {SEG_OUT_DIR}/dataset.yaml {DRIVE_SEG}/dataset.yaml 2>/dev/null || true

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Make sure we have a base yolo11n-seg weight
import urllib.request
base_weights = Path("/content/yolo11n-seg.pt")
if not base_weights.exists():
    url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n-seg.pt"
    urllib.request.urlretrieve(url, base_weights)

model = YOLO(str(base_weights))
results = model.train(
    data=f"{SEG_OUT_DIR}/dataset.yaml",
    epochs=30,
    imgsz=640,
    batch=-1,
    device=0 if USE_GPU else "cpu",
    project="/content/runs",
    name="seg",
    exist_ok=True,
    patience=10,
    workers=4,
    pretrained=True,
)
print(f"\nBest model: {results.save_dir}/weights/best.pt")

## 7. Save trained weights to Drive

In [ ]:
import shutil
from pathlib import Path

# Find the latest run
best = Path("/content/runs/seg/weights/best.pt")
if best.exists():
    dst = Path(DRIVE_ROOT) / "models" / "yolo11n-seg-pointsx.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, dst)
    print(f"Saved: {dst}")
else:
    print("No best.pt found — training may not have completed.")

---
## ✅ Done

You now have `yolo11n-seg-pointsx.pt` trained on perfect synthetic silhouettes. Use it together with your pose model in `MeasurementPipeline` for accurate body-width extraction.